# GWNet v22 — PEMS08
**Target: MAE~12, beat MD-GRTN (13.114) on all metrics**

## What gave test MAE=13.626 (v20-5)
```
seq_len=18, kernel_size=4, d_time=128, gcn_order=4, adp_emb=20
d_model=96, d_skip=512, n_blocks=8 (dilations [1,2,4,8,1,2,4,8])
epochs=55, patience=20  ← KILLED IT
```

## What killed it
`CosineAnnealingWarmRestarts(T_0=50)` restarted LR to 1e-3 at ep50.
Val MAE bounced, patience=20 triggered at ep55. Model was mid-convergence.

## v22 changes (minimal, surgical)
1. **Keep ALL winning architecture** — seq_len, kernel_size, d_time, gcn_order, adp_emb
2. **d_model: 96→128** — 33% more capacity, 7.8 GB headroom on T4
3. **Single OneCycleLR** — no restart, no conflict, smooth decay
4. **epochs=200, patience=50** — let it actually converge
5. **Huber loss δ=1.0** — smoother gradients than pure MAE
6. **Learnable skip weights** — model learns which temporal scales matter


In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} OK')

set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
class Config:
    # ── Paths ──────────────────────────────────────────────────────
    data_path    = "/kaggle/input/datasets/piyush1718s/pems08/PEMS08.npz"
    adj_csv_path = "/kaggle/input/datasets/piyush1718s/pems08csv/PEMS08.csv"
    best_path    = "gwnet_v22_best.pt"

    # ── Dataset (KEEP from v20-5 — these worked) ───────────────────
    num_nodes   = 170
    in_features = 3
    seq_len     = 18     # KEEP — 18 worked, 1.5h context
    pred_len    = 12
    feature_idx = 0
    train_ratio = 0.7
    val_ratio   = 0.1

    # ── Model (KEEP winning params, only bump d_model) ─────────────
    d_model     = 128    # ★ 96→128, +33% capacity, 7.8GB headroom
    d_skip      = 512    # KEEP
    d_end       = 512    # KEEP
    d_time      = 128    # KEEP — worked well
    kernel_size = 4      # KEEP — worked well
    adp_emb     = 20     # KEEP — worked well
    gcn_order   = 4      # KEEP — worked well (safe with d=128 on T4)
    n_supports  = 3      # fwd + bwd + adaptive
    dropout     = 0.10   # slight reduction from 0.15
    # 8 blocks with dilations that gave good receptive field in v20-5
    DILATIONS   = [1, 2, 4, 8, 1, 2, 4, 8]

    # ── Training — THE fix ────────────────────────────────────────
    batch_size   = 64
    lr           = 2e-3   # OneCycleLR max_lr (higher = faster warmup)
    epochs       = 200    # ★ was 55 — model was killed mid-convergence
    patience     = 50     # ★ was 20 — too aggressive
    weight_decay = 1e-3

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config: d={cfg.d_model} d_skip={cfg.d_skip} d_time={cfg.d_time}')
print(f'  seq={cfg.seq_len} kernel={cfg.kernel_size} gcn_order={cfg.gcn_order}')
print(f'  blocks={len(cfg.DILATIONS)} dilations={cfg.DILATIONS}')
print(f'  epochs={cfg.epochs} patience={cfg.patience} lr={cfg.lr} | {device}')

In [ ]:
def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    data = raw['data'].astype(np.float32)   # (T, N, 3)
    T, N, F = data.shape
    print(f'Data: {data.shape}')
    mean_np = data.mean(axis=0)
    std_np  = data.std(axis=0) + 1e-8
    data_norm = (data - mean_np) / std_np

    # Sparse Gaussian-kernel adjacency (real road edges only)
    A_raw = np.zeros((N, N), dtype=np.float32)
    try:
        df = pd.read_csv(cfg.adj_csv_path, header=None, skiprows=1)
        df.columns = ['from', 'to', 'cost']
        df[['from','to']] = df[['from','to']].astype(int)
        df['cost'] = df['cost'].astype(float)
        sigma = df['cost'].std()
        for _, r in df.iterrows():
            i, j, c = int(r['from']), int(r['to']), float(r['cost'])
            if i < N and j < N:
                w = float(np.exp(-(c**2)/(sigma**2+1e-8)))
                A_raw[i,j] = w;  A_raw[j,i] = w
        print(f'Adj: nnz={(A_raw>0).sum()} avg_deg={(A_raw>0).sum()/N:.1f}')
    except Exception as e:
        print(f'Adj fallback: {e}')
        np.fill_diagonal(A_raw, 1.0)
    return data_norm, mean_np, std_np, A_raw


class TrafficDataset(Dataset):
    def __init__(self, data_norm, seq_len, pred_len, feature_idx,
                 split_start=0, split_end=None):
        self.data     = data_norm
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.feat_idx = feature_idx
        T = len(data_norm)
        split_end = split_end or T
        self.indices = list(range(split_start, split_end - seq_len - pred_len + 1))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i   = self.indices[idx]
        x   = self.data[i : i+self.seq_len].copy()
        y   = self.data[i+self.seq_len : i+self.seq_len+self.pred_len,
                        :, self.feat_idx].copy()
        # tod: 5-min slot [0,287]; dow: day of week [0,6]
        tod = np.array([(i+t) % 288      for t in range(self.seq_len)], dtype=np.int64)
        dow = np.array([((i+t)//288) % 7 for t in range(self.seq_len)], dtype=np.int64)
        return (torch.from_numpy(x), torch.from_numpy(y),
                torch.from_numpy(tod), torch.from_numpy(dow))


def build_dataloaders(cfg):
    set_seed()
    data_norm, mean_np, std_np, A_raw = load_pems08(cfg)
    T  = len(data_norm)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(data_norm=data_norm, seq_len=cfg.seq_len,
                 pred_len=cfg.pred_len, feature_idx=cfg.feature_idx)
    dl_tr = DataLoader(TrafficDataset(**ds_kw, split_start=0,  split_end=t1),
                       shuffle=True, **kw)
    dl_va = DataLoader(TrafficDataset(**ds_kw, split_start=t1, split_end=t2),
                       shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset(**ds_kw, split_start=t2, split_end=T),
                       shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} Val={len(dl_va.dataset)} Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_raw

print('Data utilities ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# DIFFUSION GCN — identical to v20-5 (it worked)
# ═══════════════════════════════════════════════════════════
class DiffusionGCN(nn.Module):
    def __init__(self, d_in, d_out, n_supports=3, order=2, dropout=0.1):
        super().__init__()
        total_in = d_in * (1 + n_supports * order)
        self.mlp   = nn.Linear(total_in, d_out)
        self.drop  = nn.Dropout(dropout)
        self.order = order

    def forward(self, x, supports):  # x: (B*S, N, d)
        hs = [x]
        for A in supports:
            h = x
            for _ in range(self.order):
                h = torch.einsum('nm,bmd->bnd', A, h)
                hs.append(h)
        return self.drop(self.mlp(torch.cat(hs, dim=-1)))


# ═══════════════════════════════════════════════════════════
# WAVE BLOCK — identical to v20-5
# ═══════════════════════════════════════════════════════════
class WaveBlock(nn.Module):
    def __init__(self, d_model, d_skip, kernel_size, dilation,
                 n_supports, gcn_order, dropout):
        super().__init__()
        self.dilation    = dilation
        self.kernel_size = kernel_size
        conv_kw = dict(kernel_size=(1, kernel_size), dilation=(1, dilation))
        self.filter_conv = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gate_conv   = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gcn         = DiffusionGCN(d_model, d_model, n_supports, gcn_order, dropout)
        self.bn          = nn.BatchNorm2d(d_model)
        self.drop        = nn.Dropout(dropout)
        self.skip_conv   = nn.Conv2d(d_model, d_skip,  (1, 1))
        self.res_conv    = nn.Conv2d(d_model, d_model, (1, 1))

    def forward(self, x, supports):  # x: (B, d, N, S)
        residual = x
        pad   = (self.kernel_size - 1) * self.dilation
        x_pad = F.pad(x, [pad, 0])
        f = torch.tanh   (self.filter_conv(x_pad))
        g = torch.sigmoid(self.gate_conv  (x_pad))
        x = self.drop(f * g)
        B, d, N, S = x.shape
        xg = x.permute(0,3,2,1).reshape(B*S, N, d)
        xg = self.gcn(xg, supports)
        x  = xg.reshape(B,S,N,d).permute(0,3,2,1)
        x  = self.bn(x)
        skip = self.skip_conv(x)
        x    = self.res_conv(x) + residual
        return x, skip


# ═══════════════════════════════════════════════════════════
# GWNET v22
# Key changes vs v20-5:
#   d_model: 96 -> 128
#   skip_weights: uniform sum -> learnable
#   Everything else identical to the model that gave 13.626
# ═══════════════════════════════════════════════════════════
class GWNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        N = cfg.num_nodes

        # Static adjacency buffers
        self.register_buffer('A_fwd', torch.zeros(N, N))
        self.register_buffer('A_bwd', torch.zeros(N, N))

        # Adaptive adjacency
        self.E1 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)
        self.E2 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)

        # Input projection + node embedding
        self.start_conv = nn.Conv2d(cfg.in_features, cfg.d_model, (1, 1))
        self.node_emb   = nn.Parameter(torch.randn(1, cfg.d_model, N, 1) * 0.01)

        # Temporal embeddings (KEEP — key contributor in v20-5)
        self.tod_emb   = nn.Embedding(288, cfg.d_time)
        self.dow_emb   = nn.Embedding(7,   cfg.d_time)
        self.time_proj = nn.Linear(cfg.d_time * 2, cfg.d_model)

        # WaveBlocks — same dilations as v20-5
        self.blocks = nn.ModuleList([
            WaveBlock(cfg.d_model, cfg.d_skip, cfg.kernel_size, d,
                      cfg.n_supports, cfg.gcn_order, cfg.dropout)
            for d in cfg.DILATIONS
        ])
        self.n_blocks = len(cfg.DILATIONS)

        # ★ Learnable skip weights — one per block
        # Softmax-normalised so total contribution stays bounded
        # Init to uniform; model learns which temporal scales matter
        self.skip_w = nn.Parameter(torch.ones(self.n_blocks))

        # Output
        self.end_conv1 = nn.Conv2d(cfg.d_skip, cfg.d_end,    (1, 1))
        self.end_conv2 = nn.Conv2d(cfg.d_end,  cfg.pred_len, (1, 1))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        nn.init.normal_(self.tod_emb.weight, std=0.01)
        nn.init.normal_(self.dow_emb.weight, std=0.01)

    def set_adj(self, A_raw):
        A = torch.FloatTensor(A_raw).to(self.A_fwd.device)
        D_f = A.sum(1, keepdim=True).clamp(min=1e-8)
        D_b = A.T.sum(1, keepdim=True).clamp(min=1e-8)
        self.A_fwd.copy_(A   / D_f)
        self.A_bwd.copy_(A.T / D_b)

    def get_supports(self):
        A_adp = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)
        return [self.A_fwd, self.A_bwd, A_adp]

    def forward(self, x, _A=None, tod=None, dow=None):  # x:(B,S,N,F)
        x = x.permute(0, 3, 2, 1)            # (B,F,N,S)
        x = self.start_conv(x) + self.node_emb

        # Temporal embedding — broadcast over N
        if tod is not None and dow is not None:
            te = torch.cat([self.tod_emb(tod), self.dow_emb(dow)], dim=-1)  # (B,S,2*dt)
            te = self.time_proj(te).permute(0,2,1).unsqueeze(2)             # (B,d,1,S)
            x  = x + te

        supports = self.get_supports()
        skips = []
        for blk in self.blocks:
            x, skip = blk(x, supports)
            skips.append(skip)          # each: (B, d_skip, N, S)

        # Learnable weighted sum of last-step skips from each block
        w = F.softmax(self.skip_w, dim=0)                     # (n_blocks,)
        skip_agg = sum(w[i] * skips[i][..., -4:].mean(-1, keepdim=True)
                       for i in range(self.n_blocks))          # (B,d_skip,N,1)

        out = F.relu(self.end_conv1(F.relu(skip_agg)))        # (B,d_end,N,1)
        out = self.end_conv2(out)                              # (B,pred_len,N,1)
        return out.squeeze(-1)                                 # (B,pred_len,N)


# Sanity check
set_seed()
model = GWNet(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GWNet v22 | Parameters: {total:,}')
print(f'  Blocks: {model.n_blocks} | dilations: {cfg.DILATIONS}')
print(f'  gcn_order={cfg.gcn_order} | kernel={cfg.kernel_size} | d_time={cfg.d_time}')
with torch.no_grad():
    dx  = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    dt  = torch.randint(0, 288, (2, cfg.seq_len)).to(device)
    dw  = torch.randint(0, 7,   (2, cfg.seq_len)).to(device)
    out = model(dx, tod=dt, dow=dw)
print(f'Output: {out.shape}  expect [2, 12, 170]')
if torch.cuda.is_available():
    print(f'GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB / 15 GB')
    torch.cuda.empty_cache()

In [ ]:
def masked_mae(pred, true):
    mask = (true.abs() > 0.0).float()
    return (torch.abs(pred - true) * mask).sum() / (mask.sum() + 1e-8)


def huber_loss(pred, true, delta=1.0):
    """
    Huber loss (smooth L1):
    - Quadratic for |err| < delta (smooth gradients near zero)
    - Linear for |err| >= delta (robust to outliers)
    delta=1.0 in normalised space is a well-balanced choice.
    """
    err  = torch.abs(pred - true)
    loss = torch.where(err < delta,
                       0.5 * err**2,
                       delta * (err - 0.5 * delta))
    return loss.mean()


def masked_rmse(pred, true):
    mask = (true.abs() > 0.0).float()
    return torch.sqrt(((pred - true)**2 * mask).sum() / (mask.sum() + 1e-8))


def masked_mape(pred, true, low_thresh=10.0):
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1:
        return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred - true) / (true.abs() + 1e-8)) * mask).sum() / mask.sum() * 100


print('Metrics ready.')

In [ ]:
set_seed()
dl_train, dl_val, dl_test, mean_np, std_np, A_raw = build_dataloaders(cfg)

mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)
model.set_adj(A_raw)

print(f'Train batches: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}')
print(f'A_fwd nnz: {int((model.A_fwd > 0).sum())}')

In [ ]:
scaler = torch.amp.GradScaler('cuda')


def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total = 0.0
    for x, y, tod, dow in loader:
        x   = x.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)
        tod = tod.to(device, non_blocking=True)
        dow = dow.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(x, tod=tod, dow=dow)
            loss = huber_loss(pred, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()   # per-batch step for smooth LR curve
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x, y, tod, dow in loader:
        x   = x.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)
        tod = tod.to(device, non_blocking=True)
        dow = dow.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(x, tod=tod, dow=dow)
        pd_ = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        yd_ = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append (masked_mae (pd_, yd_).item())
        rmses.append(masked_rmse(pd_, yd_).item())
        mapes.append(masked_mape(pd_, yd_).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)


print('Train / eval ready.')

In [ ]:
set_seed()
model = GWNet(cfg).to(device)
model.set_adj(A_raw)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# ═══════════════════════════════════════════════════════════
# SINGLE OneCycleLR — replaces the 3-scheduler mess in v20-5
#
# v20-5 had: warmup(LambdaLR) + CosineAnnealingWarmRestarts(T_0=50)
#            + ReduceLROnPlateau — all fighting each other.
# CosineAnnealingWarmRestarts reset LR to 1e-3 at ep50.
# Val MAE jumped, patience=20 triggered → dead at ep55.
#
# OneCycleLR: clean 10% warmup → smooth cosine to lr/10000.
# No restarts. No plateau. One schedule, all the way.
# ═══════════════════════════════════════════════════════════
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr           = cfg.lr,
    epochs           = cfg.epochs,
    steps_per_epoch  = len(dl_train),
    pct_start        = 0.10,
    anneal_strategy  = 'cos',
    div_factor       = 10.0,
    final_div_factor = 1000.0
)

best_val_mae  = float('inf')
best_val_rmse = float('inf')
best_val_mape = float('inf')
patience_cnt  = 0
history       = {'train_loss':[], 'val_mae':[], 'val_rmse':[], 'val_mape':[]}

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GWNet v22 | {params:,} params')
print(f'  d={cfg.d_model} d_skip={cfg.d_skip} d_time={cfg.d_time}')
print(f'  gcn_order={cfg.gcn_order} kernel={cfg.kernel_size} seq={cfg.seq_len}')
print(f'  OneCycleLR max_lr={cfg.lr} | epochs={cfg.epochs} patience={cfg.patience}')
print(f'  Huber loss δ=1.0 | Learnable skip weights')
print(f'Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*70)

for epoch in range(1, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, scheduler, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, device, mean_flow, std_flow)

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae  = val_mae
        best_val_rmse = val_rmse
        best_val_mape = val_mape
        patience_cnt  = 0
        torch.save(model.state_dict(), cfg.best_path)
        print(f'  Best saved -> {cfg.best_path}')
        tag = '  <- best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    beat   = ' *** BEAT BASELINE ***' if val_mae < 13.114 else ''
    if epoch % 5 == 0 or tag:
        print(f'Ep {epoch:03d} | Loss={train_loss:.4f} | '
              f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% '
              f'lr={lr_now:.1e}{tag}{beat}')

print(f'\nBest Val -> MAE={best_val_mae:.3f} RMSE={best_val_rmse:.3f} MAPE={best_val_mape:.2f}%')
print(f'Baseline:   MAE=13.114   RMSE=22.623   MAPE=8.471%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_title('Train Loss (Huber δ=1.0)')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.626, color='orange', ls=':', label='v20-5 test=13.626')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE')
axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE')
axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves_v22.png', dpi=150)
plt.show()

# Show learned block importance
w = F.softmax(model.skip_w, dim=0).detach().cpu().numpy()
print('\nLearned skip weights (which WaveBlock temporal scales matter):')
for i, (wi, dil) in enumerate(zip(w, cfg.DILATIONS)):
    bar = '|' * int(wi * 160)
    print(f'  Block {i+1} dil={dil:2d}: {wi:.4f}  {bar}')

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x, y, tod, dow in loader:
        x   = x.to(device); y   = y.to(device)
        tod = tod.to(device); dow = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x, tod=tod, dow=dow)
        pd_ = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        yd_ = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pd_.cpu())
        all_true.append(yd_.cpu())

    P = torch.cat(all_pred, dim=0)
    Y = torch.cat(all_true, dim=0)
    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1e-8))).mean().item()*100

    print('\n' + '='*62)
    print('  TEST RESULTS — GWNet v22 — all 12 steps')
    print('='*62)
    print(f'  MAE  : {mae:.3f}   v20-5: 13.626   baseline: 13.114   d={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}   v20-5: 23.070   baseline: 22.623   d={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%  v20-5:  7.674%  baseline:  8.471%  d={mape-8.471:+.3f}%')
    all_beat = (mae < 13.114) and (rmse < 22.623) and (mape < 8.471)
    print(f'\n  All 3 baselines beaten: {"YES" if all_beat else "partial"}')
    print('='*62)
    return mae, rmse, mape

mae, rmse, mape = paper_style_eval(model, dl_test, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x, y, tod, dow in loader:
        x   = x.to(device); y   = y.to(device)
        tod = tod.to(device); dow = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x, tod=tod, dow=dow)
        pd_ = pred.float()*std_flow[None,None,:]+mean_flow[None,None,:]
        yd_ = y.float()   *std_flow[None,None,:]+mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append (masked_mae (pd_[:,h,:], yd_[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pd_[:,h,:], yd_[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pd_[:,h,:], yd_[:,h,:]).item())

    print(f"\n{'Horizon':>14} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>9}")
    print('-'*52)
    for h, lbl in zip([2,5,11], ['3-step(15m)','6-step(30m)','12-step(60m)']):
        m = {k: np.mean(v) for k,v in buf[h].items()}
        print(f"{lbl:>14} | {m['mae']:>8.3f} | {m['rmse']:>8.3f} | {m['mape']:>8.2f}%")

horizon_eval(model, dl_test, device, mean_flow, std_flow)